# Recognising the genre and style of an artwork from a photo

## 1. Context and goal

The goal is to offer museum visitors a simple application: the visitor quickly takes a photo of an artwork, and the application immediately tells them its **genre** (portrait, landscape, religious scene, still life…) and its **style** (Impressionism, Baroque, Cubism…). The classification therefore needs to be both fast and reliable.

## 2. Results of the previous models

The best results so far come from a **ResNet18** pre-trained on ImageNet, whose last layer (`layer4`) was fine-tuned (learning rate 1e-4), on 128 × 128 pixel images with 300 images per class (see the [`results`](https://github.com/lcaquelin/Neural_Networks/tree/main/results) folder of the repository).

| Task | Number of classes | Test accuracy |
|---|---|---|
| Genre | 11 | **58.6%** |
| Style | 27 | **36.6%** |

The corresponding confusion matrices are `results/figures/resnet_confusion_test.png` (genre) and `results/figures/resnet_style_confusion_test.png` (style).

### A level of reliability that is not enough for a museum

In practice, the model is wrong roughly **once in two for style** and **more than once in three for genre**. A model that is wrong once in two or once in three times can undermine the credibility of the application. A museum is an institution that is expected to be reliable, whose mission is to share and promote culture: an application that regularly gives wrong information would go against that mission.

## 3. Why is the accuracy so low?

### Comparison with photo recognition

These scores may be surprising, because the same kind of network achieves excellent results on photographs, with many more classes. On **ImageNet** (about 1.2 million photos split into **1,000 classes**):

| Model | Top-1 accuracy | Top-5 accuracy |
|---|---|---|
| ResNet18 | 69.8% | 89.1% |
| ResNet50 | 76.1% (up to 80.9% with an improved training recipe) | 93 to 95% |

With 1,000 classes, a random guess would only have a 0.1% chance of being right, against 3.7% for our 27 styles and 9% for our 11 genres. The problem with artworks is therefore not the number of classes: it is the very nature of the categories.

### A style is not just what you see

An ImageNet class is defined by a **visible object**: a dog, a car, a cup. An artistic style, on the contrary, is often defined by a **period, themes, intentions or a historical context**, and not only by an appearance. The network, however, only judges on visual criteria: colours, texture, brushwork, composition.

The style confusion matrix shows this clearly:

- Styles defined by a **visible technique** are well recognised: Pointillism (37 correct answers out of 45), Ukiyo-e (34), Color Field (31), Minimalism (29), Pop Art (27).
- Styles defined by a **content or an idea** are poorly recognised, with less than 15% correct answers: Expressionism (4), Abstract Expressionism (4), Art Nouveau (3), Symbolism (6), Realism (7).
- **Symbolism** is defined by its themes (myths, dreams, mystery), which are invisible to a neural network: it is predicted as Romanticism 15 times, against 6 correct answers.
- **Realism** refers to an intention (depicting reality) rather than an appearance: 20th-century realisms (New Realism, Contemporary Realism) have a modern palette and brushwork, and are confused with Impressionism and Post-Impressionism rather than with 19th-century Realism.
- **Art Nouveau** is a very heterogeneous style (posters, illustration, decoration): its predictions are scattered across almost every class.

The same phenomenon exists for genres: **genre painting** (scenes of everyday life) is confused with religious painting (10 times, against 12 correct answers), because what separates them is the subject depicted, not the way it is painted.

Other factors add to this: sometimes ambiguous labels (the same artist can be classified in several styles depending on the artwork), a 128-pixel resolution that erases part of the painter's brushwork, and limited training (300 images per class, only the last layer of the network fine-tuned).

## 4. Ways to improve

### Solution 1: group close styles that the model confuses

The first consequence is to group styles that the model confuses, **if and only if** this grouping makes sense, that is, if these styles are artistically close. The artistic criterion always comes first.

- **Example of a justified grouping**: Early Renaissance, High Renaissance, Northern Renaissance and Mannerism form a "Renaissance" group. They belong to the same period, and the model confuses them (Northern Renaissance is predicted as Early Renaissance 11 times). Likewise, Baroque and Rococo, or the three Cubisms (Cubism, Analytical Cubism, Synthetic Cubism).
- **Counter-example**: Romanticism and Realism are often confused by the model (14 times), but Realism was built in opposition to Romanticism. They therefore stay separate.
- Conversely, a style that the model distinguishes well (such as Pointillism) stays in its own class.

This work reduces the 27 styles to 18 groups. By grouping the predictions of the current matrix, the accuracy would rise from about 37% to about 46%.

### Solution 2: use a more powerful model (CNN-RNN)

The second solution is to use a more powerful model, able to recognise styles better. After a search in the literature and on GitHub, an interesting piece of work is the [Wikiart_Classification_CNN-RNN](https://github.com/Karvy-Singh/Wikiart_Classification_CNN-RNN) repository, which reaches better scores on the same WikiArt dataset:

| Task | Top-1 accuracy | Top-5 accuracy |
|---|---|---|
| Style (27 classes) | **67.4%** | 95.5% |
| Genre (10 classes) | **79.2%** | 98.6% |
| Artist (23 classes) | 90.9% | 98.4% |

These scores remain lower than those of ResNet on photos, but they are almost twice as good as ours for style.

**Why would it work better?** The model combines a CNN (ResNet50) with a **recurrent neural network (RNN)**. In language processing, an RNN reads a sentence word by word while remembering the previous words. Here, it does the same with the image: the feature map produced by the CNN is read row by row and then column by column, in both directions, by two bidirectional GRUs. The model therefore remembers the pixels (or groups of pixels) it has already read. Each area of the image is no longer treated independently: it is interpreted by taking the image as a whole into account, its composition, its alignments and its rhythm. In a classic ResNet, on the contrary, the last step averages all the areas of the image and loses this spatial organisation.

Note: part of the gain also comes from the rest of the training recipe (ResNet50 instead of ResNet18, 384-pixel images, the full dataset, data augmentations such as MixUp and CutMix, weighted sampling). This should be kept in mind when measuring what the RNN part really brings.

### Solution 3: combine both approaches

The third solution is to combine both approaches to maximise the results: train the CNN-RNN model on the style groups from solution 1. The more powerful model reduces errors, and the grouping removes the remaining confusions between artistically close styles.

### Solution 4: suggest several possible answers (top 3)

Finally, instead of displaying a single answer, the application can suggest **the three most likely styles (or genres)**, to avoid giving a wrong answer presented as certain. Indeed, when the model is wrong, it usually names a close category and makes few big mistakes: Northern Renaissance predicted as Early Renaissance, Baroque predicted as Rococo, landscape predicted as cityscape…

The CNN-RNN model confirms this: the correct answer is among its top 5 suggestions in 95.5% of cases for style and 98.6% of cases for genre. An answer such as "Baroque, possibly Rococo or Mannerism" remains honest, and it also has educational value: it introduces the visitor to the movements related to the artwork they are looking at.